In [1]:
import pandas as pd

# Set pandas display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 100)        # Set the display width to 1000 characters
pd.set_option('display.max_colwidth', None) # Allow the full content of each column to be displayed

In [2]:
df_final_audit = pd.read_csv('./comprehensive_audit_final.csv')

In [3]:
import os, json, time
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── 설정 ─────────────────────────────────────────────────
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

df_final_audit = pd.read_csv('./sfscore_df.csv', encoding='utf-8-sig')
all_vars = [col.replace('원본_', '') for col in df_final_audit.columns if col.startswith('원본_')]

np.random.seed(42)
df_sample = df_final_audit.sample(n=100).reset_index(drop=True)
print(f"샘플: {len(df_sample)}건")

# ── 변형 A: 현행 프롬프트 ────────────────────────────────
SYSTEM_A = (
    "당신은 보험 언더라이팅 전문가입니다. "
    "주어진 의료 소견서에서 아래 변수들을 정확히 추출하여 JSON 형식으로만 반환하십시오. "
    "다른 텍스트는 출력하지 마십시오."
)
def user_A(sogyeon):
    return (
        f"다음 의료 소견서에서 아래 변수들을 추출하여 JSON으로 반환하십시오.\n"
        f"변수 목록: {all_vars}\n\n소견서:\n{sogyeon}\n\n"
        f'응답 형식: {{"변수명": "값", ...}}'
    )

# ── 변형 B: JSON 스키마 명시 ──────────────────────────────
SYSTEM_B = (
    "당신은 보험 언더라이팅 전문가입니다. "
    "주어진 의료 소견서에서 지정된 변수들을 추출하여 아래 JSON 스키마에 맞춰 반환하십시오. "
    "숫자는 단위 없이 숫자만, 범주형은 원문 그대로 반환하십시오. "
    "값이 없으면 null로 반환하십시오. 다른 텍스트는 출력하지 마십시오."
)
def user_B(sogyeon):
    schema = {v: "숫자 또는 범주값" for v in all_vars}
    return (
        f"소견서:\n{sogyeon}\n\n"
        f"다음 스키마에 맞춰 JSON으로 반환하십시오:\n{json.dumps(schema, ensure_ascii=False)}"
    )

# ── 변형 C: few-shot 예시 포함 ────────────────────────────
FEW_SHOT_EXAMPLE = """예시 입력:
수축기 혈압 130 mmHg, 체질량지수 24.5 kg/m²입니다. 공복혈당 95 mg/dL로 정상입니다.
[CLINICAL_DATA_RECORD]
- 최종 수축기 혈압: 130.0 mmHg
- 체질량지수: 24.5 kg/m²

예시 출력:
{"최종 수축기 혈압": "130.0", "체질량지수": "24.5"}"""

SYSTEM_C = (
    "당신은 보험 언더라이팅 전문가입니다. "
    "주어진 의료 소견서에서 아래 변수들을 정확히 추출하여 JSON 형식으로만 반환하십시오. "
    "다른 텍스트는 출력하지 마십시오.\n\n"
    f"{FEW_SHOT_EXAMPLE}"
)
def user_C(sogyeon):
    return (
        f"다음 의료 소견서에서 아래 변수들을 추출하여 JSON으로 반환하십시오.\n"
        f"변수 목록: {all_vars}\n\n소견서:\n{sogyeon}\n\n"
        f'응답 형식: {{"변수명": "값", ...}}'
    )

# ── 추출 함수 ────────────────────────────────────────────
def extract_one(idx_row, system_prompt, user_fn):
    idx, row = idx_row
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_fn(row['합성_소견서'])}
            ],
            temperature=0,
            max_tokens=1024,
        )
        raw = response.choices[0].message.content.strip()
        raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        extracted = json.loads(raw)
        extracted['개인아이디'] = row['개인아이디']
        return idx, extracted, None
    except Exception as e:
        return idx, {'개인아이디': row['개인아이디'], '_error': str(e)}, str(e)

# ── Sf 계산 ──────────────────────────────────────────────
CLINICAL_TOLERANCE = {
    '최종 수축기 혈압': 5.0, '최종 이완기 혈압': 5.0,
    '체질량지수': 0.5, '허리둘레': 1.0, '공복혈당': 5.0,
    '당화혈색소': 0.2, '총콜레스테롤': 5.0,
    'HDL-콜레스테롤': 3.0, '중성지방': 5.0,
    'LDL-콜레스테롤(직접검사)': 5.0,
}
clinical_cols = ['최종 수축기 혈압', '최종 이완기 혈압', '공복혈당', '체질량지수']
w_c, w_l = 0.68, 0.32
n_life = len(all_vars) - len(clinical_cols)

def parse_num(val):
    try:
        return float(str(val).replace(' ', '').replace(',', '.')
                     .split('m')[0].split('k')[0].split('%')[0])
    except:
        return None

def calc_sf(orig_row, gpt_row):
    scores = []
    for var in all_vars:
        orig = orig_row.get(f'원본_{var}')
        recon = gpt_row.get(var)
        w = (w_c / len(clinical_cols)) if var in clinical_cols else (w_l / n_life)
        o, r = parse_num(orig), parse_num(recon)
        if o is not None and r is not None:
            tol = CLINICAL_TOLERANCE.get(var, 5.0)
            scores.append((1.0 if abs(o - r) <= tol else 0.0) * w)
        else:
            scores.append((1.0 if str(orig).strip() == str(recon).strip() else 0.0) * w)
    return sum(scores)

# ── 3가지 변형 실행 ──────────────────────────────────────
variants = {
    'A_현행': (SYSTEM_A, user_A),
    'B_스키마': (SYSTEM_B, user_B),
    'C_fewshot': (SYSTEM_C, user_C),
}

sf_results = {}

for name, (sys_prompt, usr_fn) in variants.items():
    print(f"\n▶ 변형 {name} 실행 중...")
    results = [None] * len(df_sample)
    errors = 0
    done = 0

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(extract_one, (i, row), sys_prompt, usr_fn): i
                   for i, row in df_sample.iterrows()}
        for future in as_completed(futures):
            idx, result, err = future.result()
            results[idx] = result
            done += 1
            if err:
                errors += 1
            if done % 20 == 0:
                print(f"  {done}/100건 완료 (오류: {errors}건)", flush=True)

    df_gpt = pd.DataFrame(results)
    df_gpt_valid = df_gpt[~df_gpt.get('_error', pd.Series([None]*len(df_gpt))).notna()].copy() \
        if '_error' in df_gpt.columns else df_gpt.copy()
    df_merged = df_sample.merge(df_gpt_valid, on='개인아이디', how='inner')
    df_merged['Sf'] = df_merged.apply(lambda row: calc_sf(row, row), axis=1)
    sf_results[name] = {
        'mean_sf': df_merged['Sf'].mean(),
        'sf_1.0': (df_merged['Sf'] == 1.0).mean() * 100,
        'sf_gte_0.9': (df_merged['Sf'] >= 0.9).mean() * 100,
        'errors': errors,
        'n': len(df_merged)
    }
    print(f"  완료. 평균 Sf={df_merged['Sf'].mean():.4f}")

# ── 결과 비교 ────────────────────────────────────────────
print("\n" + "=" * 65)
print("프롬프트 민감도 분석 결과 (추출 프롬프트 변형, N=100)")
print("=" * 65)
print(f"{'변형':<15} {'평균 Sf':>10} {'Sf≥0.9(%)':>12} {'Sf=1.0(%)':>12} {'오류':>6}")
print("-" * 65)
for name, r in sf_results.items():
    print(f"{name:<15} {r['mean_sf']:>10.4f} {r['sf_gte_0.9']:>11.1f}% {r['sf_1.0']:>11.1f}% {r['errors']:>6}")

pd.DataFrame(sf_results).T.to_csv('prompt_sensitivity_result.csv', encoding='utf-8-sig')
print("\n결과 저장: prompt_sensitivity_result.csv")

샘플: 100건

▶ 변형 A_현행 실행 중...
  20/100건 완료 (오류: 0건)
  40/100건 완료 (오류: 0건)
  60/100건 완료 (오류: 0건)
  80/100건 완료 (오류: 0건)
  100/100건 완료 (오류: 0건)
  완료. 평균 Sf=0.8992

▶ 변형 B_스키마 실행 중...
  20/100건 완료 (오류: 0건)
  40/100건 완료 (오류: 0건)
  60/100건 완료 (오류: 0건)
  80/100건 완료 (오류: 0건)
  100/100건 완료 (오류: 0건)
  완료. 평균 Sf=0.8853

▶ 변형 C_fewshot 실행 중...
  20/100건 완료 (오류: 0건)
  40/100건 완료 (오류: 0건)
  60/100건 완료 (오류: 0건)
  80/100건 완료 (오류: 0건)
  100/100건 완료 (오류: 0건)
  완료. 평균 Sf=0.9102

프롬프트 민감도 분석 결과 (추출 프롬프트 변형, N=100)
변형                   평균 Sf    Sf≥0.9(%)    Sf=1.0(%)     오류
-----------------------------------------------------------------
A_현행                0.8992        57.0%         0.0%      0
B_스키마               0.8853        39.0%         0.0%      0
C_fewshot           0.9102        85.0%         0.0%      0

결과 저장: prompt_sensitivity_result.csv
